# Proyecto final · Clasificación de prendas con Fashion-MNIST

**Estudiante:** Christian Gabriel Villafuerte Aguilar  
**Tema:** Redes neuronales artificiales · Computer Vision  
**Fecha:** 20 de noviembre de 2026

## Objetivo
Entrenar y comparar **3 configuraciones diferentes** de redes neuronales para clasificar imágenes de **Fashion-MNIST**, incluyendo obligatoriamente la capa `Flatten`, una visualización **2D** de lo aprendido y la métrica **accuracy**.

## Diseño experimental
- **Datos:** 60,000 ejemplos de entrenamiento de origen y 10,000 ejemplos de prueba. Del entrenamiento se reservan 6,000 para validación: 54,000 entrenan y 6,000 validan.
- **Preprocesamiento:** imágenes de 28×28 píxeles en escala de grises, normalizadas entre 0 y 1.
- **Modelo 1:** `Flatten → Dense(128) → Dense(64) → Dense(10)`.
- **Modelo 2:** `Flatten → Dense(256) → Dropout(0.25) → Dense(128) → Dropout(0.15) → Dense(10)`.
- **Modelo 3:** `Flatten → Dense(128) → Dense(2, rep_2d) → Dense(10)`.
- **Métricas:** `accuracy` y `loss` en entrenamiento, validación y prueba.
- **Gráficas en 2D:** imágenes originales, imagen aplanada, curvas accuracy/loss, comparación de modelos, dispersión de 2 neuronas (`rep_2d`), matriz de confusión y predicciones.

**Basado en los ejercicios proporcionados en clase:** `02_Computer_Vision_MNIST_Intro.ipynb` y `01 - Validation & Callback.ipynb`.

> **Ejecución de resultados:** las mediciones de accuracy, gráficas de entrenamiento y conclusiones numéricas se generan ejecutando las celdas en orden. No se presentan resultados de entrenamiento que no se hayan medido. Al terminar, guarda este archivo `.ipynb` en VS Code para conservar las salidas y poder entregarlo.


## 0. Resultados verificables de la estructura de las redes

La siguiente salida está calculada matemáticamente a partir de las capas solicitadas (incluyendo sesgos). **No representa accuracy, pérdida ni un entrenamiento realizado**.

In [1]:
# Comprobación REAL de arquitectura (no es un resultado de entrenamiento)
# Parámetros en Dense(m -> n) = m*n + n
configuraciones_estructurales = {
    "Modelo 1 - Base": [(784, 128), (128, 64), (64, 10)],
    "Modelo 2 - Dropout": [(784, 256), (256, 128), (128, 10)],
    "Modelo 3 - Latente 2D": [(784, 128), (128, 2), (2, 10)],
}
for nombre, capas in configuraciones_estructurales.items():
    parametros = sum(n_in * n_out + n_out for n_in, n_out in capas)
    print(f"{nombre:<25} {parametros:>7,} parámetros")
print("Imagen: 28 × 28 × 1 → Flatten: 784 características")
print("Salida: 10 neuronas / 10 categorías")

Modelo 1 - Base           109,386 parámetros
Modelo 2 - Dropout        235,146 parámetros
Modelo 3 - Latente 2D     100,768 parámetros
Imagen: 28 × 28 × 1 → Flatten: 784 características
Salida: 10 neuronas / 10 categorías


## Preparación y ejecución

Este notebook está diseñado para **Visual Studio Code (extensiones Python + Jupyter)** o **Google Colab**. Selecciona un kernel de **Python 3.11 o 3.12** en VS Code y usa **Run All / Ejecutar todas**.

La siguiente celda comprueba que estén disponibles los módulos requeridos y, si falta alguno, intenta instalarlo automáticamente en el kernel. La instalación necesita conexión a Internet. Si aparece una indicación de reiniciar el kernel, reinícialo y ejecuta todas las celdas otra vez.


In [ ]:
# Preparación de dependencias (solo instala paquetes faltantes).
import importlib.util
import subprocess
import sys

requisitos = {
    "tensorflow": "tensorflow>=2.16,<2.22",
    "tensorflow_datasets": "tensorflow-datasets>=4.9",
    "sklearn": "scikit-learn>=1.3",
    "matplotlib": "matplotlib>=3.7",
    "pandas": "pandas>=2.0",
    "numpy": "numpy>=1.24",
}
faltantes = [paquete for modulo, paquete in requisitos.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    print("Instalando bibliotecas faltantes:", faltantes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
else:
    print("Todas las bibliotecas están disponibles.")


Instalando bibliotecas faltantes: ['tensorflow>=2.16,<2.22', 'tensorflow-datasets>=4.9']


## 1. Importar bibliotecas y configurar reproducibilidad

Trabajaremos con TensorFlow/Keras para construir las redes y `tensorflow_datasets` para cargar los datos tal como se solicita. Se fija una semilla para reducir variaciones entre ejecuciones.

In [ ]:
# %pip install -q tensorflow-datasets

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

SEMILLA = 42
BATCH_SIZE = 32
EPOCHS = 15
RESULTADOS_DIR = "resultados_fashion_mnist"
os.makedirs(RESULTADOS_DIR, exist_ok=True)

tf.keras.utils.set_random_seed(SEMILLA)
np.random.seed(SEMILLA)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

## 2. Cargar Fashion-MNIST con TensorFlow Datasets

Al igual que en el código de la consigna, se usan `tfds.load`, `as_supervised=True` y `batch_size=32`. Se subdivide **solo el conjunto de entrenamiento** en 90 % entrenamiento y 10 % validación. El conjunto de prueba se reserva hasta la comparación final, evitando elegir un modelo por su resultado en prueba.

In [ ]:
train_raw, val_raw, test_raw = tfds.load(
    "fashion_mnist",
    split=["train[:90%]", "train[90%:]", "test"],
    shuffle_files=True,
    as_supervised=True,
    batch_size=BATCH_SIZE,
)

class_names = [
    "Camiseta/top", "Pantalón", "Suéter", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Tenis", "Bolso", "Botín",
]

print("Entrenamiento: 54,000 imágenes")
print("Validación:     6,000 imágenes")
print("Prueba:        10,000 imágenes")

imagenes_muestra, etiquetas_muestra = next(iter(train_raw))
print("Forma de un batch:", imagenes_muestra.shape)
print("Etiquetas:", etiquetas_muestra[:10].numpy())

## 3. Visualizar las imágenes originales (2D)

Cada imagen está formada por una matriz de **28 × 28 píxeles** en escala de grises. `imshow()` permite ver las imágenes en 2D antes de convertirlas en vectores.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(imagenes_muestra[i].numpy().squeeze(), cmap="gray")
    ax.set_title(class_names[int(etiquetas_muestra[i])])
    ax.axis("off")
plt.suptitle("Fashion-MNIST: ejemplos de las 10 clases", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Normalizar los píxeles a [0, 1]

Se divide cada valor entre 255. Se mantienen los tensores con forma `(28, 28, 1)` porque **`Flatten` se aplicará dentro de cada red neuronal**, no antes de entrenar. El barajado se realiza solo en el conjunto de entrenamiento.

In [ ]:
def normalizar(imagenes, etiquetas):
    imagenes = tf.cast(imagenes, tf.float32) / 255.0
    return imagenes, etiquetas

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (
    train_raw
    .map(normalizar, num_parallel_calls=AUTOTUNE)
    .shuffle(500, seed=SEMILLA, reshuffle_each_iteration=True)
    .prefetch(AUTOTUNE)
)
val_ds = val_raw.map(normalizar, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_raw.map(normalizar, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

x_batch, y_batch = next(iter(train_ds))
print("Mínimo:", float(tf.reduce_min(x_batch)), "Máximo:", float(tf.reduce_max(x_batch)))
print("Forma de entrada:", x_batch.shape)

## 5. Comprobar el funcionamiento de `Flatten`

La imagen de 28 × 28 × 1 pasa a un vector de **784 características**. Esto es igual a lo que se hizo en el notebook `02_Computer_Vision_MNIST_Intro.ipynb`, donde se observó la imagen y después el vector aplanado. Aquí mostramos ambos resultados con gráficos 2D.

In [ ]:
flatten_demo = tf.keras.layers.Flatten()
vector_batch = flatten_demo(x_batch)

print("Antes de Flatten:", tuple(x_batch.shape))
print("Después de Flatten:", tuple(vector_batch.shape))
assert vector_batch.shape[1] == 784

fig, axes = plt.subplots(2, 1, figsize=(13, 5), gridspec_kw={"height_ratios": [4, 1]})
axes[0].imshow(x_batch[0].numpy().squeeze(), cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Imagen 2D original: 28 × 28")
axes[0].axis("off")

axes[1].imshow(vector_batch[0].numpy().reshape(1, 784), cmap="gray", aspect="auto", vmin=0, vmax=1)
axes[1].set_title("Vector Flatten: 1 × 784")
axes[1].set_yticks([])
axes[1].set_xlabel("Índice del píxel")
plt.tight_layout()
plt.show()

## 6. Definir tres configuraciones de red neuronal

Todas usan **`Input → Flatten → capas Dense → Softmax`**, pérdida `sparse_categorical_crossentropy` y métrica `accuracy`.

| Modelo | Arquitectura después de `Flatten` | Propósito |
|---|---|---|
| **Modelo 1 · Base** | Dense(128, relu) → Dense(64, relu) → Dense(10, softmax) | Referencia como la práctica de MNIST |
| **Modelo 2 · Profundo + Dropout** | Dense(256, relu) → Dropout(0.25) → Dense(128, relu) → Dropout(0.15) → Dense(10, softmax) | Probar más capacidad y regularización |
| **Modelo 3 · Representación 2D** | Dense(128, relu) → Dense(2, linear, `rep_2d`) → Dense(10, softmax) | Aprender dos coordenadas internas para graficar en 2D |

La capa de salida tiene **10 neuronas** por las diez categorías de prendas, y `softmax` entrega probabilidades de clase. La tercera red puede tener menos accuracy porque comprime toda la información a solo dos valores.

In [ ]:
def crear_modelo(nombre):
    capas = [
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Flatten(name="flatten"),
    ]

    if nombre == "Modelo 1 - Base":
        capas += [
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(64, activation="relu"),
        ]
    elif nombre == "Modelo 2 - Dropout":
        capas += [
            tf.keras.layers.Dense(256, activation="relu"),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(0.15),
        ]
    elif nombre == "Modelo 3 - Latente 2D":
        capas += [
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(2, activation="linear", name="rep_2d"),
        ]
    else:
        raise ValueError(f"Configuración no reconocida: {nombre}")

    capas.append(tf.keras.layers.Dense(10, activation="softmax"))
    modelo = tf.keras.Sequential(capas, name=nombre.replace(" ", "_").replace("-", ""))
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo

configuraciones = [
    "Modelo 1 - Base",
    "Modelo 2 - Dropout",
    "Modelo 3 - Latente 2D",
]

for nombre in configuraciones:
    modelo_temporal = crear_modelo(nombre)
    print(f"{nombre}: {modelo_temporal.count_params():,} parámetros")
# Comprobar dimensiones y parámetros de las tres arquitecturas.
for nombre in configuraciones:
    m = crear_modelo(nombre)
    assert m.input_shape[1:] == (28, 28, 1)
    assert m.output_shape[-1] == 10
    assert any(isinstance(c, tf.keras.layers.Flatten) for c in m.layers)
print("Validación de las tres arquitecturas: correcta.")


## 7. Graficar curvas de aprendizaje

Se adapta la función `plot_learning_curve(history)` de las prácticas anteriores. Cada figura muestra dos gráficos 2D: `accuracy` y `loss`, tanto para entrenamiento como para validación.

In [ ]:
def plot_learning_curve(history, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    epocas = np.arange(1, len(history.history["accuracy"]) + 1)

    axes[0].plot(epocas, history.history["accuracy"], label="Entrenamiento", marker="o", markersize=3)
    axes[0].plot(epocas, history.history["val_accuracy"], label="Validación", marker="o", markersize=3)
    axes[0].set(title="Accuracy", xlabel="Época", ylabel="Accuracy", ylim=(0, 1))

    axes[1].plot(epocas, history.history["loss"], label="Entrenamiento", marker="o", markersize=3)
    axes[1].plot(epocas, history.history["val_loss"], label="Validación", marker="o", markersize=3)
    axes[1].set(title="Pérdida (loss)", xlabel="Época", ylabel="Loss")

    for ax in axes:
        ax.legend()
        ax.grid(True, alpha=0.3)

    fig.suptitle(titulo, fontsize=14)
    plt.tight_layout()
    nombre_archivo = (titulo.lower().replace(" ", "_").replace("-", "")).strip("_")
    fig.savefig(os.path.join(RESULTADOS_DIR, f"curvas_{nombre_archivo}.png"), dpi=160, bbox_inches="tight")
    plt.show()

## 8. Entrenar las tres redes y registrar su accuracy

`ModelCheckpoint` guarda la mejor versión de cada arquitectura según `val_accuracy`. `EarlyStopping` detiene el entrenamiento cuando la validación deja de mejorar. El mismo conjunto de validación y el mismo límite de **15 épocas** se utilizan para las tres configuraciones.

**Ejecuta la siguiente celda completa.** Este paso entrena los modelos realmente, por lo que sus resultados dependerán de la ejecución y del hardware.

In [ ]:
os.makedirs("models", exist_ok=True)
modelos = {}
historias = {}
filas_resumen = []

for i, nombre in enumerate(configuraciones, start=1):
    print("\n" + "=" * 65)
    print(f"ENTRENANDO {i}/3: {nombre}")
    print("=" * 65)

    modelo = crear_modelo(nombre)
    ruta_modelo = f"models/fashion_mnist_modelo_{i}.keras"

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=3,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=ruta_modelo,
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
        ),
    ]

    history = modelo.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=callbacks,
        verbose=1,
    )

    # Usamos exactamente los pesos de la mejor época de validación.
    modelo = tf.keras.models.load_model(ruta_modelo)
    modelos[nombre] = modelo
    historias[nombre] = history

    mejor_epoca = int(np.argmax(history.history["val_accuracy"]) + 1)
    mejor_val_accuracy = float(np.max(history.history["val_accuracy"]))

    filas_resumen.append({
        "Modelo": nombre,
        "Parámetros": modelo.count_params(),
        "Épocas realizadas": len(history.history["accuracy"]),
        "Mejor época": mejor_epoca,
        "Accuracy validación": mejor_val_accuracy,
    })

    print(f"Mejor época: {mejor_epoca}")
    print(f"Accuracy de validación: {mejor_val_accuracy:.2%}")
    plot_learning_curve(history, nombre)

resumen_val = pd.DataFrame(filas_resumen).sort_values(
    "Accuracy validación", ascending=False
).reset_index(drop=True)
print("\nCOMPARACIÓN DE VALIDACIÓN")
print(resumen_val.to_string(index=False, formatters={"Accuracy validación": "{:.2%}".format}))

## 9. Comparar gráficamente las tres configuraciones

Las curvas y barras se construyen usando las accuracies observadas, no valores supuestos. Para **elegir el mejor modelo** se utiliza `accuracy` de validación.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for nombre in configuraciones:
    hist = historias[nombre].history
    axes[0].plot(
        np.arange(1, len(hist["val_accuracy"]) + 1),
        hist["val_accuracy"], marker="o", markersize=3, label=nombre,
    )
axes[0].set(title="Accuracy de validación por época", xlabel="Época", ylabel="Accuracy")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].bar(resumen_val["Modelo"], 100 * resumen_val["Accuracy validación"])
axes[1].set(title="Mejor accuracy de validación", ylabel="Accuracy (%)", ylim=(0, 100))
axes[1].tick_params(axis="x", rotation=20)
for i, fila in resumen_val.iterrows():
    axes[1].text(i, 100 * fila["Accuracy validación"] + 1, f"{fila['Accuracy validación']:.2%}", ha="center")

plt.tight_layout()
fig.savefig(os.path.join(RESULTADOS_DIR, "comparacion_modelos.png"), dpi=160, bbox_inches="tight")
plt.show()

nombre_mejor = resumen_val.iloc[0]["Modelo"]
mejor_modelo = modelos[nombre_mejor]
print(f"Modelo elegido según validación: {nombre_mejor}")

## 10. Evaluación final con el conjunto de prueba

**Solo después de entrenar y seleccionar el modelo**, se mide `accuracy` con datos que los modelos no vieron durante el entrenamiento ni para validación. Se reporta la accuracy de prueba de las tres arquitecturas para comparar y la mejor se determina por validación, no por el test.

In [ ]:
resultados_prueba = []
for nombre in configuraciones:
    metricas = modelos[nombre].evaluate(test_ds, verbose=0, return_dict=True)
    resultados_prueba.append({
        "Modelo": nombre,
        "Accuracy test": float(metricas["accuracy"]),
        "Loss test": float(metricas["loss"]),
    })

df_test = pd.DataFrame(resultados_prueba)
comparacion_final = resumen_val.merge(df_test, on="Modelo", how="left")
print(comparacion_final.to_string(index=False, formatters={
    "Accuracy validación": "{:.2%}".format,
    "Accuracy test": "{:.2%}".format,
    "Loss test": "{:.4f}".format,
}))

comparacion_final.to_csv(os.path.join(RESULTADOS_DIR, "comparacion_fashion_mnist.csv"), index=False)
print(f"\nModelo seleccionado: {nombre_mejor}")
print("Accuracy de prueba:", f"{df_test.set_index('Modelo').loc[nombre_mejor, 'Accuracy test']:.2%}")

## 11. Graficar el espacio latente en 2D (SIN PCA)

Esta parte reproduce la idea del notebook de **MNIST Intro**: la tercera red incluye la capa `Dense(2, activation='linear', name='rep_2d')`. Se extraen los valores de sus dos neuronas para 3,000 imágenes de validación y se representan en un plano XY. **Cada punto es una prenda** y su color representa la clase real.

*Importante:* `Flatten` transforma cada imagen en un vector de 784 componentes; la capa `rep_2d` aprende una representación distinta de **dos componentes**, que sí puede representarse como un conjunto de puntos en 2D.

In [ ]:
modelo_2d = modelos["Modelo 3 - Latente 2D"]
extractor = tf.keras.Model(
    inputs=modelo_2d.inputs,
    outputs=modelo_2d.get_layer("rep_2d").output,
)

n_visual = 3000
x_lista, y_lista = [], []
total = 0
for imagenes, etiquetas in val_ds:
    x_lista.append(imagenes.numpy())
    y_lista.append(etiquetas.numpy())
    total += len(etiquetas)
    if total >= n_visual:
        break

x_visual = np.concatenate(x_lista, axis=0)[:n_visual]
y_visual = np.concatenate(y_lista, axis=0)[:n_visual]
puntos2d = extractor.predict(x_visual, batch_size=256, verbose=0)
print("Coordenadas del espacio latente:", puntos2d.shape)

fig, ax = plt.subplots(figsize=(10, 7))
scatter_plot = ax.scatter(
    puntos2d[:, 0], puntos2d[:, 1],
    c=y_visual, cmap="tab10", vmin=-0.5, vmax=9.5,
    s=9, alpha=0.65,
)
ax.set_title("Espacio latente 2D aprendido por el MLP (sin PCA)")
ax.set_xlabel("Neurona de representación 1")
ax.set_ylabel("Neurona de representación 2")
ax.grid(True, linestyle="--", alpha=0.3)
cbar = fig.colorbar(scatter_plot, ax=ax, ticks=np.arange(10))
cbar.ax.set_yticklabels(class_names)
cbar.set_label("Clase real")
plt.tight_layout()
fig.savefig(os.path.join(RESULTADOS_DIR, "espacio_latente_2D.png"), dpi=160, bbox_inches="tight")
plt.show()

## 12. Matriz de confusión y predicciones del modelo seleccionado

Se visualiza en qué clases acierta o se confunde el mejor modelo. Las etiquetas y predicciones se calculan **en el mismo recorrido de `test_ds`**, evitando desalineaciones por el orden del dataset.

In [ ]:
y_reales, y_predichas = [], []
imagenes_ejemplo, clases_ejemplo, predicciones_ejemplo = [], [], []

for imagenes, etiquetas in test_ds:
    probabilidades = mejor_modelo(imagenes, training=False).numpy()
    predicciones = np.argmax(probabilidades, axis=1)
    y_reales.extend(etiquetas.numpy().tolist())
    y_predichas.extend(predicciones.tolist())

    faltantes = 10 - len(imagenes_ejemplo)
    if faltantes > 0:
        imagenes_ejemplo.extend(imagenes[:faltantes].numpy())
        clases_ejemplo.extend(etiquetas[:faltantes].numpy().tolist())
        predicciones_ejemplo.extend(predicciones[:faltantes].tolist())

fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay.from_predictions(
    y_reales,
    y_predichas,
    labels=np.arange(10),
    display_labels=class_names,
    cmap="Blues",
    values_format="d",
    xticks_rotation=45,
    ax=ax,
)
ax.set_title("Matriz de confusión de prueba: " + nombre_mejor)
plt.tight_layout()
fig.savefig(os.path.join(RESULTADOS_DIR, "matriz_confusion.png"), dpi=160, bbox_inches="tight")
plt.show()

print(classification_report(
    y_reales, y_predichas, target_names=class_names, digits=3, zero_division=0,
))

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(imagenes_ejemplo[i].squeeze(), cmap="gray")
    correcto = clases_ejemplo[i] == predicciones_ejemplo[i]
    ax.set_title(
        f"Real: {class_names[clases_ejemplo[i]]}\nPred: {class_names[predicciones_ejemplo[i]]}",
        color="green" if correcto else "red", fontsize=9,
    )
    ax.axis("off")
plt.suptitle("Predicciones sobre imágenes no vistas")
plt.tight_layout()
fig.savefig(os.path.join(RESULTADOS_DIR, "predicciones_prueba.png"), dpi=160, bbox_inches="tight")
plt.show()

## 13. Conclusión generada a partir del experimento

La siguiente celda redacta un resumen con el **modelo elegido**, su accuracy de validación y de prueba. Para un reporte de clase, complementa esta conclusión explicando la diferencia entre las arquitecturas y qué tan separadas aparecen las categorías en el espacio latente 2D.

In [ ]:
fila_ganadora = comparacion_final.loc[comparacion_final["Modelo"] == nombre_mejor].iloc[0]
print("CONCLUSIONES")
print("- Se utilizaron imágenes de Fashion-MNIST normalizadas al intervalo [0, 1].")
print("- Se entrenaron tres redes neuronales con la capa Flatten, softmax y métrica accuracy.")
print(f"- La arquitectura seleccionada por validación fue: {nombre_mejor}.")
print(f"- Su mejor accuracy de validación fue {fila_ganadora['Accuracy validación']:.2%}.")
print(f"- Su accuracy final de prueba fue {fila_ganadora['Accuracy test']:.2%}.")
print("- La tercera arquitectura aprendió una representación de dos neuronas,")
print("  visualizada como un gráfico de dispersión en dos dimensiones sin usar PCA.")
print("- Los resultados pueden variar ligeramente entre ejecuciones y dispositivos.")

## Evidencias que se guardan automáticamente

La ejecución del notebook genera los archivos de resultados dentro de `resultados_fashion_mnist/`: curvas, tabla de comparación `.csv`, visualización de la representación 2D, matriz de confusión y ejemplos de predicciones.

**Para entregar:** ejecuta todas las celdas y guarda el **mismo archivo `.ipynb`** desde VS Code (Ctrl+S) para que las salidas queden incrustadas. La tabla que aparece en las secciones 9–10 contiene las accuracies observadas sobre validación y prueba, no estimaciones.


## 15. Resumen reproducible para entregar

Esta celda produce un archivo de texto breve usando **solo los resultados reales calculados arriba**. El resumen permite comprobar el modelo con mejor validación y su accuracy de prueba, sin copiar números manualmente.

In [ ]:
mejor_fila = comparacion_final.loc[
    comparacion_final["Modelo"] == nombre_mejor
].iloc[0]

resumen_entrega = f"""PRÁCTICA: FASHION-MNIST CON TRES REDES NEURONALES
Nombre: Christian Gabriel Villafuerte Aguilar

Datos: 54,000 entrenamiento / 6,000 validación / 10,000 prueba.
Arquitecturas: Base, Dropout y representación latente 2D (todas con Flatten).
Métrica comparada: accuracy.

{comparacion_final.to_string(index=False)}

Mejor modelo en validación: {nombre_mejor}
Accuracy de validación del mejor modelo: {mejor_fila['Accuracy validación']:.4%}
Accuracy de prueba del mejor modelo: {mejor_fila['Accuracy test']:.4%}

Se evaluaron tres configuraciones con la misma partición de datos.
La gráfica de dos dimensiones corresponde a las activaciones de Dense(2, rep_2d).
"""
from pathlib import Path
ruta_resumen = Path(RESULTADOS_DIR) / "reporte_final.txt"
ruta_resumen.write_text(resumen_entrega, encoding="utf-8")
print(resumen_entrega)
print("Resumen guardado en:", ruta_resumen.resolve())
